# NETCDF Historical Splitting
This notebook (in python as we don't deal directly with tARget here) creates a separate netcdf file for historical (1980-2013) data only, which helps in calculating historical pixel IVT limits only.

In [3]:
import xarray as xa
import cftime
import pandas as pd
from glob import glob

In [2]:
wusd3 = pd.read_csv('../../wusd3_gcms.csv', index_col=0)

for i in range(1011, 1201, 20):
    wusd3.loc[len(wusd3)] = ['cesm2-le', 'CESM2-le', str(i), '365_day']

wusd3

,Model,Model Name,Member,Calendar
0,access-cm2,ACCESS-CM2,r5i1p1f1,standard
1,canesm5,CanESM5,r1i1p2f1,365_day
2,cesm2,CESM2,r11i1p1f1,365_day
3,cnrm-esm2-1,CNRM-ESM2-1,r1i1p1f2,standard
4,ec-earth3,EC-Earth3,r1i1p1f1,standard
5,ec-earth3-veg,EC-Earth3-Veg,r1i1p1f1,standard
6,era5,ERA5,NaN,standard
7,fgoals-g3,FGOALS-g3,r1i1p1f1,365_day
8,giss-e2-1-g,GISS-E2-1-G,r1i1p1f2,365_day
9,miroc6,MIROC6,r1i1p1f1,standard


In [6]:
## 40 year files for pixel_ivt_limit calculations
for i in range(len(wusd3.Model)):
    model = wusd3.iat[i, 0]
    member = wusd3.iat[i, 2]
    cal = wusd3.iat[i, 3]

    print(f"{model}/{member}")
    sdate = cftime.datetime(1980, 9, 1, calendar=cal)
    edate = cftime.datetime(2014, 8, 30, calendar=cal) if cal=='360_day' else cftime.datetime(2014, 8, 31, calendar=cal)

    ivtfile = f'/glade/derecho/scratch/tcorrie/regrids/ivt.daily.{model}.{member}.d01_regridded.nc'
    ivt = xa.open_dataset(ivtfile, use_cftime=True)
    ivtcut = ivt.sel(time=slice(sdate,edate))
    dunits = sdate.strftime('days since %Y-%m-%d')
    # Encoding time units is necessary for tARget to get the dates right.
    ivtcut.to_netcdf(f'/glade/derecho/scratch/tcorrie/regrids/ivt.daily.{model}.{member}.d01_regridded_tARgetivtcalc.nc', encoding={'time': {'units': dunits}})

access-cm2/r5i1p1f1
canesm5/r1i1p2f1
cesm2/r11i1p1f1
cnrm-esm2-1/r1i1p1f2
ec-earth3/r1i1p1f1
ec-earth3-veg/r1i1p1f1
era5/nan
fgoals-g3/r1i1p1f1
giss-e2-1-g/r1i1p1f2
miroc6/r1i1p1f1
mpi-esm1-2-hr/r3i1p1f1
mpi-esm1-2-hr/r7i1p1f1
mpi-esm1-2-lr/r7i1p1f1
noresm2-mm/r1i1p1f1
taiesm1/r1i1p1f1
ukesm1-0-ll/r2i1p1f2
cesm2-le/1011
cesm2-le/1031
cesm2-le/1051
cesm2-le/1071
cesm2-le/1091
cesm2-le/1111
cesm2-le/1131
cesm2-le/1151
cesm2-le/1171
cesm2-le/1191


In [5]:
## ATTENTION: this cell isn't used in the current version of my analysis. If you run it, it won't break anything, but
#    you'll be making extra netcdf files that may not be needed.

## Split into individual years
for i in range(len(wusd3.Model)):
    model = wusd3.iat[i, 0]
    member = wusd3.iat[i, 2]
    cal = wusd3.iat[i, 3]

    if i != 6 and i<=15:
        syear = 1980
        eyear = 2100
    elif i==6:
        syear = 1950
        eyear = 2025
    elif i>=16:
        syear = 1850
        eyear = 2100
    print(f"{model}/{member}")
   

    ivtfile = f'/glade/derecho/scratch/tcorrie/regrids/ivt.daily.{model}.{member}.d01_regridded.nc'
    ivt = xa.open_dataset(ivtfile, use_cftime=True)
    for yr in range(syear, eyear):
        sdate = cftime.datetime(yr, 9, 1, calendar=cal)
        edate = cftime.datetime(yr+1, 8, 30, calendar=cal) if cal=='360_day' else cftime.datetime(yr+1, 8, 31, calendar=cal)
        try:
            ivtcut = ivt.sel(time=slice(sdate,edate))
        except:
            ivtcut = ivt.sel(time=slice(sdate,None))
        ivtcut.to_netcdf(f'/glade/derecho/scratch/tcorrie/regrids/temptARget/ivt.daily.{model}.{member}.d01_regridded.{yr}.nc')

access-cm2/r5i1p1f1
canesm5/r1i1p2f1
cesm2/r11i1p1f1
cnrm-esm2-1/r1i1p1f2
ec-earth3/r1i1p1f1
ec-earth3-veg/r1i1p1f1
era5/nan
fgoals-g3/r1i1p1f1
giss-e2-1-g/r1i1p1f2
miroc6/r1i1p1f1
mpi-esm1-2-hr/r3i1p1f1
mpi-esm1-2-hr/r7i1p1f1
mpi-esm1-2-lr/r7i1p1f1
noresm2-mm/r1i1p1f1
taiesm1/r1i1p1f1
ukesm1-0-ll/r2i1p1f2
cesm2-le/1011
cesm2-le/1031
cesm2-le/1051
cesm2-le/1071
cesm2-le/1091
cesm2-le/1111
cesm2-le/1131
cesm2-le/1151
cesm2-le/1171
cesm2-le/1191
